In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

# CONFIG

In [ ]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ranking Loss Configuration
USE_RANKING_LOSS = True
RANKING_MARGIN = 0
BCE_WEIGHT = 0  # Weight for auxiliary BCE loss
# MAX_PAIRS_PER_RULE = 5000*4  # Limit pairs per rule to control training size

In [ ]:
# Set seeds
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [ ]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [ ]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
print(df.head())
print(df.columns.tolist())

In [ ]:
def create_rule_pairs(dataframe):
    """Create positive-negative pairs within each rule with sampling"""
    pairs = []
    rules = []
    targets = []
    
    # Group by rule
    for rule in dataframe['rule'].unique():
        rule_data = dataframe[dataframe['rule'] == rule]
        
        # Get positive and negative examples for this rule
        positives = []
        negatives = []
        
        # Add main training examples
        train_rule_data = rule_data[rule_data.get('is_test_data', False) == False]
        rule_positives = train_rule_data[train_rule_data['label'] == 1]['text'].tolist()
        rule_negatives = train_rule_data[train_rule_data['label'] == 0]['text'].tolist()
        positives.extend(rule_positives)
        negatives.extend(rule_negatives)


        # Add positive examples from the rule
        for col in ['positive_example_1', 'positive_example_2']:
            if col in rule_data.columns:
                pos_examples = (rule_data['rule'] + ' [SEP] ' + rule_data[col]).tolist()
                pos_examples = list(set(pos_examples))  # deduplication
                positives.extend(pos_examples)
        
        # Add negative examples from the rule
        for col in ['negative_example_1', 'negative_example_2']:
            if col in rule_data.columns:
                neg_examples = (rule_data['rule'] + ' [SEP] ' + rule_data[col]).tolist()
                neg_examples = list(set(neg_examples))  # deduplication
                negatives.extend(neg_examples)
        
        # Remove duplicates
        positives = list(set(positives))
        negatives = list(set(negatives))
        
        # Create all possible pairs for this rule
        rule_pairs = []
        for pos in positives:
            for neg in negatives:
                rule_pairs.append((pos, neg))
                
        MAX_PAIRS_PER_RULE = min(len(pos),len(neg))*5
        
        # Sample pairs if too many
        if len(rule_pairs) > MAX_PAIRS_PER_RULE:
            np.random.seed(SEED)
            sampled_indices = np.random.choice(len(rule_pairs), MAX_PAIRS_PER_RULE, replace=False)
            rule_pairs = [rule_pairs[i] for i in sampled_indices]
        
        # Add sampled pairs
        pairs.extend(rule_pairs)
        rules.extend([rule] * len(rule_pairs))
        targets.extend([1] * len(rule_pairs))  # 1 means positive should rank higher than negative
        
        print(f"Rule '{rule}': {len(positives)} pos, {len(negatives)} neg -> {len(rule_pairs)} pairs")
    
    return pairs, rules, targets

In [ ]:
class JigsawRankingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, df_test=None):
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # Create rule-aware pairs
        df_combined = dataframe.copy()
        if df_test is not None:
            df_test_copy = df_test.copy()
            df_test_copy["text"] = df_test_copy["rule"] + " [SEP] " + df_test_copy["body"]
            df_test_copy["label"] = 0
            df_test_copy["is_test_data"] = True  # Add this flag
            df_combined = pd.concat([df_combined, df_test_copy], ignore_index=True)
            
        self.pairs, self.rules, self.targets = create_rule_pairs(df_combined)
        print(f'Created {len(self.pairs)} ranking pairs from rule-aware data')

    def __len__(self): 
        return len(self.pairs)

    def __getitem__(self, idx):
        pos_text, neg_text = self.pairs[idx]
        target = self.targets[idx]  # Should be 1 for MarginRankingLoss
        
        # Tokenize positive example
        pos_enc = self.tokenizer(
            pos_text, padding='max_length', truncation=True, 
            max_length=self.max_len, return_tensors="pt"
        )
        
        # Tokenize negative example
        neg_enc = self.tokenizer(
            neg_text, padding='max_length', truncation=True, 
            max_length=self.max_len, return_tensors="pt"
        )
        
        return {
            'pos_input_ids': pos_enc['input_ids'].squeeze(0),
            'pos_attention_mask': pos_enc['attention_mask'].squeeze(0),
            'neg_input_ids': neg_enc['input_ids'].squeeze(0),
            'neg_attention_mask': neg_enc['attention_mask'].squeeze(0),
            'target': torch.tensor(target, dtype=torch.float),  # 1 for pos > neg
            'rule': self.rules[idx]
        }

# Keep original dataset for validation on individual samples
class JigsawDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len,df=None,df_test=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        if df is not None:
            extras=add_data(df)
            self.texts+= extras[0]
            self.labels+= extras[1]
            print('Added additional data of size from train examples',len(extras[1]))
        if df_test is not None:
            extras=add_data(df_test)
            self.texts+= extras[0]
            self.labels+= extras[1]
            print('Added additional data of size from test examples',len(extras[1]))

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        tmp= list(set(tmp))#deduplication
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.2)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch_ranking(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    ranking_criterion = nn.MarginRankingLoss(margin=RANKING_MARGIN)
    bce_criterion = nn.BCEWithLogitsLoss()
    
    for batch in tqdm(loader):
        optimizer.zero_grad()
        
        # Forward pass for positive examples
        pos_ids = batch["pos_input_ids"].to(DEVICE)
        pos_mask = batch["pos_attention_mask"].to(DEVICE)
        pos_logits = model(pos_ids, pos_mask)
        
        # Forward pass for negative examples
        neg_ids = batch["neg_input_ids"].to(DEVICE)
        neg_mask = batch["neg_attention_mask"].to(DEVICE)
        neg_logits = model(neg_ids, neg_mask)
        
        # Ranking targets (positive should rank higher than negative)
        targets = batch["target"].to(DEVICE)
        
        # Margin ranking loss
        ranking_loss = ranking_criterion(pos_logits, neg_logits, targets)
        
        # Optional: Add auxiliary BCE loss for regularization
        if BCE_WEIGHT > 0:
            # Treat positive examples as label 1, negative as label 0
            pos_bce = bce_criterion(pos_logits, torch.ones_like(pos_logits))
            neg_bce = bce_criterion(neg_logits, torch.zeros_like(neg_logits))
            bce_loss = (pos_bce + neg_bce) / 2
            total_loss_batch = ranking_loss + BCE_WEIGHT * bce_loss
        else:
            total_loss_batch = ranking_loss
        
        total_loss_batch.backward()
        optimizer.step()
        total_loss += total_loss_batch.item()
    
    return total_loss / len(loader)

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        optimizer.step()
        # scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate_simple(model, loader):
    """Simple validation on individual samples like original BCE approach"""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model(input_ids, mask)
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
    
    auc = roc_auc_score(targets, preds)
    return auc, np.array(preds)

def validate_ranking(model, ranking_loader, individual_loader):
    """Validate using both ranking pairs and individual samples"""
    model.eval()
    
    # Validate on ranking pairs
    ranking_loss = 0
    ranking_criterion = nn.MarginRankingLoss(margin=RANKING_MARGIN)
    
    with torch.no_grad():
        for batch in ranking_loader:
            pos_ids = batch["pos_input_ids"].to(DEVICE)
            pos_mask = batch["pos_attention_mask"].to(DEVICE)
            pos_logits = model(pos_ids, pos_mask)
            
            neg_ids = batch["neg_input_ids"].to(DEVICE)
            neg_mask = batch["neg_attention_mask"].to(DEVICE)
            neg_logits = model(neg_ids, neg_mask)
            
            targets = batch["target"].to(DEVICE)
            loss = ranking_criterion(pos_logits, neg_logits, targets)
            ranking_loss += loss.item()
    
    # Validate on individual samples for AUC calculation
    preds, targets = [], []
    with torch.no_grad():
        for batch in individual_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model(input_ids, mask)
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
    
    auc = roc_auc_score(targets, preds)
    avg_ranking_loss = ranking_loss / len(ranking_loader)
    
    return auc, avg_ranking_loss, np.array(preds)

def validate(model, loader):
    model.eval()
    preds, targets = [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
    auc = roc_auc_score(targets, preds)
    val_loss = total_loss / len(loader)
    return auc, val_loss ,np.array(preds)

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    all_preds = []
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    
    for fold, (tr_idx, val_idx) in enumerate(folds.split(df, df["label"])):
        print(f"\n===== Fold {fold + 1} =====")
        df_test = pd.read_csv(test_path)
        
        if USE_RANKING_LOSS:
            # Prepare dataframes for ranking pairs (TRAINING ONLY)
            train_df = df.iloc[tr_idx].copy()
            
            # Create ranking training dataset
            train_ranking_ds = JigsawRankingDataset(train_df, tokenizer, MAX_LEN, df_test=df_test)
            train_ranking_loader = DataLoader(train_ranking_ds, batch_size=BATCH_SIZE//2, shuffle=True)
            
            # Simple individual validation dataset (NO PAIRS)
            val_ds = JigsawDataset(df.iloc[val_idx]['text'].tolist(), df.iloc[val_idx]['label'].tolist(), tokenizer, MAX_LEN)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
            
        else:
            # Original approach
            train_ds = JigsawDataset(df.iloc[tr_idx]['text'].tolist(), df.iloc[tr_idx]['label'].tolist(), tokenizer, MAX_LEN,df=df,df_test=df_test)
            val_ds = JigsawDataset(df.iloc[val_idx]['text'].tolist(), df.iloc[val_idx]['label'].tolist(), tokenizer, MAX_LEN)
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize model
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        
        for name, param in model.named_parameters():
            if name.startswith('base.'):
                param.requires_grad = False
        
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        best_auc = 0
        
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            
            if USE_RANKING_LOSS:
                # Train with ranking loss, validate simply
                train_loss = train_one_epoch_ranking(model, train_ranking_loader, optimizer, None)
                val_auc, val_preds = validate_simple(model, val_loader)
                print(f"Train Loss: {train_loss:.4f}, Val AUC: {val_auc:.4f}")
            else:
                # Train with BCE loss
                train_loss = train_one_epoch(model, train_loader, optimizer, None)
                val_auc, val_loss, val_preds = validate(model, val_loader)
                print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            
            if epoch <= 6:
                for name, param in model.named_parameters(): 
                    if name.startswith('base.') and (str(11-epoch) in name):
                        param.requires_grad = True
            
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}.bin")
    
        all_preds.append(pd.Series(val_preds))

In [ ]:
# all_truths=[]
# all_rules=[]
# folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
# for fold, (tr_idx, val_idx) in enumerate(folds.split(df, df["label"])):
#     # print(val_idx[0])
#     all_truths.append(df.iloc[val_idx].label)
#     all_rules.append(df.iloc[val_idx].rule)

# preddf= pd.DataFrame(columns=['preds','truths','rule'])
# preddf.preds=pd.concat(all_preds,ignore_index=True)
# preddf.rule= pd.concat(all_rules,ignore_index=True)
# preddf.truths= pd.concat(all_truths,ignore_index=True)

# print(preddf.groupby('rule').apply(lambda group: roc_auc_score(group['truths'],group['preds'])))

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"model_fold{fold}.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv